# 00 — Preparar os rótulos demográficos e dividir a base

Este notebook **não infere raça ou gênero das fotos**. Lê os códigos já presentes nos nomes UTKFace e registra sua proveniência. Os rótulos da base são estimados e revisados pelos autores, não autodeclarações nem identidade de gênero comprovada. As categorias originais são preservadas; não equivalem às categorias do IBGE.

Execute antes da geração. O ZIP fornecido contém várias árvores: usamos somente `UTKFace/` para evitar misturar cópias e recortes. Não é necessário descompactar tudo.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tcc_justica')
ROOT.mkdir(parents=True, exist_ok=True)
print('Pasta persistente:', ROOT)


## 1. Indicar o ZIP
Envie `UTK.zip` ao Drive, na pasta abaixo, ou altere o caminho. Os arquivos resultantes ficam na mesma pasta e serão usados pelos demais Colabs.


In [ ]:
ZIP_PATH = ROOT / 'UTK.zip'
assert ZIP_PATH.is_file(), f'Coloque UTK.zip em {ROOT}'


## 2. Ler rótulos, verificar imagens e remover cópias exatas
Usamos hash dos pixels decodificados. Isso remove cópias exatas, mas não identifica pessoas repetidas em fotos diferentes. Havendo IDs de pessoa, forneça `identidades.csv` com `filename,subject_id` antes de dividir. Sem isso, a divisão é por arquivo, com limitação explícita.


In [ ]:
import zipfile, re, hashlib, io
import pandas as pd
from PIL import Image
RACES = {0:'White', 1:'Black', 2:'Asian', 3:'Indian', 4:'Others'}
GENDERS = {0:'male', 1:'female'}
rows, rejected, seen = [], [], {}
imgdir = ROOT / 'dados' / 'reais'; imgdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    names = sorted(n for n in z.namelist() if n.startswith('UTKFace/') and n.lower().endswith(('.jpg','.png','.jpeg')))
    assert names, 'A pasta UTKFace/ não foi encontrada no ZIP.'
    for name in names:
        try:
            m = re.match(r'^(\d+)_(\d+)_(\d+)_', Path(name).name)
            if not m: raise ValueError('Nome sem códigos válidos')
            age, gender, race = map(int,m.groups())
            if age not in range(117) or gender not in GENDERS or race not in RACES:
                raise ValueError('Código fora do dicionário')
            raw=z.read(name)
            with Image.open(io.BytesIO(raw)) as im:
                im=im.convert('RGB'); digest=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
            if digest in seen:
                rejected.append({'filename':name,'motivo':'pixels duplicados','original':seen[digest]});continue
            seen[digest]=name
            dest=imgdir/Path(name).name;dest.write_bytes(raw)
            rows.append(dict(image_id=digest,filename=dest.name,image_path=str(dest),label=0,
                             age=age,gender=GENDERS[gender],race=RACES[race],
                             annotation_source='UTKFace_filename',dataset='UTKFace',sha256_pixels=digest))
        except Exception as e: rejected.append({'filename':name,'motivo':str(e)})
df=pd.DataFrame(rows)
assert len(df), 'Nenhuma imagem válida.'
pd.DataFrame(rejected,columns=['filename','motivo','original']).to_csv(ROOT/'exclusoes.csv',index=False)
print('Imagens válidas:',len(df),'Exclusões:',len(rejected))
display(pd.crosstab(df.race,df.gender))


## 3. Dividir antes de gerar manipulações
Treino 70%, validação 15%, teste 15%, aproximadamente. Fotos da mesma pessoa, quando identificadas nos metadados, ficam juntas. O split é estratificado por grupo quando cada unidade possui rótulo consistente. Não escolher divisão pelos resultados do teste.


In [ ]:
from sklearn.model_selection import train_test_split
ids=ROOT/'identidades.csv'
if ids.exists():
    ident=pd.read_csv(ids,dtype=str)
    assert ident.filename.is_unique and ident.subject_id.notna().all()
    df=df.merge(ident[['filename','subject_id']],on='filename',how='left',validate='one_to_one')
    assert df.subject_id.notna().all(), 'Faltam IDs de pessoa.'
    df['unit_id']=df.subject_id
    split_level='subject'
else:
    df['unit_id']=df.image_id
    split_level='file_only'
    print('LIMITAÇÃO: IDs de pessoa ausentes; fotos diferentes da mesma pessoa podem cruzar splits.')
df['stratum']=df.race+'|'+df.gender
assert df.groupby('unit_id').stratum.nunique().max()==1, 'Rótulos conflitantes por pessoa: revisar metadados.'
units=df[['unit_id','stratum']].drop_duplicates()
assert units.stratum.value_counts().min()>=10, 'Poucas unidades em algum grupo; revise a divisão.'
train,rest=train_test_split(units,test_size=.30,stratify=units.stratum,random_state=42)
val,test=train_test_split(rest,test_size=.50,stratify=rest.stratum,random_state=42)
mapping={u:s for s,part in [('train',train),('val',val),('test',test)] for u in part.unit_id}
df['split']=df.unit_id.map(mapping);df['split_level']=split_level
df.drop(columns='stratum').to_csv(ROOT/'metadata_reais.csv',index=False)
df[['image_path','race','gender']].to_csv(ROOT/'metadata.csv',index=False)
# metadata.csv serve para ferramentas antigas; não use o gerador antigo sem divisão prévia.
display(df.groupby(['split','race','gender']).size().rename('n').to_frame())


## Saídas e interpretação
`metadata_reais.csv` é a entrada do notebook 01. `metadata.csv` é apenas um formato de compatibilidade. Para outras bases, importe um CSV com rótulos documentados e caminho único; mantenha ausências como desconhecidas. Não preencha raça/gênero por inspeção visual ou por classificação automatizada.

Fonte dos códigos: https://susanqq.github.io/UTKFace/ . Uso da base: pesquisa não comercial, conforme seus autores.
